# day14_vision · 전이학습 후보 비교·선택·저장·복원

**목표:** 같은 8 epoch 예산의 후보 4개를 valid만으로 비교하고, 미리 정한 규칙으로 선택한 모델을 클래스 매핑·입력 계약과 함께 저장한 뒤 다시 불러 같은 예측을 재현한다.

**기능:**
1. 데이터·source 출처를 확인한다.
2. 후보 4개를 정의하고 학습한다(TODO D07).
3. Macro F1 → loss → 후보 순서로 선택한다(TODO D08).
4. 비교표·예측·혼동행렬·CAM을 저장한다.
5. 선택 모델과 계약을 저장하고(TODO D09) 새 모델 객체로 복원해 비교한다(TODO D10).

> 입력: `data/` · 결과: `outputs_notebook/` · 요약보고서: `day14_vision_요약보고서.md`
> 주의: `resnet18_official.py`(공식 ImageNet 가중치, CUDA 사용 코드 포함)는 이 노트북에서 실행하지 않는다.

In [1]:
# ============================================================
# 0. 라이브러리 및 경로 설정
# ============================================================

# 경로를 운영체제와 무관하게 다루는 Path 클래스
from pathlib import Path
# metrics.json·cam_index.json 저장
import json
# 수치 배열 계산
import numpy as np
# 표 데이터 처리·CSV 저장
import pandas as pd
# PyTorch: 텐서·모델 학습·저장
import torch

# BASE : 노트북이 있는 day14_vision 폴더 (같은 폴더의 engine.py 등을 import)
# DATA : 자체 포함 data / OUT : 노트북 결과 폴더
BASE = Path.cwd()
DATA = BASE / "data"
OUT = BASE / "outputs_notebook"
OUT.mkdir(exist_ok=True)

# CPU 1-thread 고정 -> 재현성
torch.set_num_threads(1)
print("DATA:", DATA.resolve())
print("OUT :", OUT.resolve())

DATA: /mnt/attach/project-files/20261105_day14/03_심화/day14_vision/data
OUT : /mnt/attach/project-files/20261105_day14/03_심화/day14_vision/outputs_notebook


## 1단계 · 데이터와 source 출처

In [2]:
# 제공 데이터셋 생성 함수
from engine import datasets

train, valid = datasets(DATA)
print("class_to_idx:", train.class_to_idx)
print("train:", torch.bincount(torch.tensor(train.targets)).tolist(), "/ valid:", torch.bincount(torch.tensor(valid.targets)).tolist())

# source Backbone 출처 기록 (ImageNet 아님, target 미사용)
provenance = json.loads((DATA / "source_provenance.json").read_text(encoding="utf-8"))
{k: provenance[k] for k in ["architecture", "imagenet", "source_train", "source_valid", "selected_epoch", "target_used"]}

class_to_idx: {'circle': 0, 'square': 1, 'triangle': 2}
train: [120, 40, 20] / valid: [30, 30, 30]


{'architecture': 'TinyShapeCNN_3-8-16_GAP4',
 'imagenet': False,
 'source_train': 240,
 'source_valid': 60,
 'selected_epoch': 48,
 'target_used': False}

## 2단계 · 후보 정의 (TODO D07)
이 순서가 동률일 때의 우선순위다. weight와 sampler는 별개 후보다.

In [3]:
# TODO D07 정답 : (strategy, balance) 후보 4개
experiments = [("head", "none"), ("partial", "none"), ("head", "weight"), ("head", "sampler")]
experiments

[('head', 'none'),
 ('partial', 'none'),
 ('head', 'weight'),
 ('head', 'sampler')]

## 3단계 · 학습과 선택 (TODO D08)
선택 키 `(macro_f1, -loss)`: F1이 크고, 같으면 loss가 작은 후보. 엄격한 `>` 비교라 완전 동률이면 먼저 나온 후보 유지.

In [4]:
# 제공 학습 골격
from engine import experiment

rows, histories, best_key = [], [], None
for strategy, balance in experiments:
    model, summary, history, prediction = experiment(DATA, strategy=strategy, balance=balance, epochs=8)
    rows.append(summary)
    history["candidate"] = strategy + "_" + balance
    histories.append(history)

    # TODO D08 정답
    key = (summary["macro_f1"], -summary["loss"])
    print(f"{strategy:8s}{balance:8s} key={key}")
    if best_key is None or key > best_key:
        best_key = key
        chosen = summary
        chosen_pred = prediction
        chosen_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

print("선택:", chosen["strategy"], chosen["balance"], "test 사용: False")

head    none     key=(1.0, -0.0126413118508127)


partial none     key=(1.0, -0.00814789914422565)


head    weight   key=(1.0, -0.00655005888806449)


head    sampler  key=(1.0, -0.006474726564354367)
선택: head sampler test 사용: False


## 4단계 · 비교표·예측·혼동행렬 저장

In [5]:
# 클래스별 리포트·혼동행렬
from sklearn.metrics import classification_report, confusion_matrix

metrics_table = pd.DataFrame(rows)
metrics_table.to_csv(OUT / "metrics.csv", index=False)
pd.concat(histories, ignore_index=True).to_csv(OUT / "history.csv", index=False)

model.load_state_dict(chosen_state)
model.eval()
chosen_pred.to_csv(OUT / "predictions.csv", index=False)

report = classification_report(chosen_pred.actual, chosen_pred.predicted, labels=[0, 1, 2], target_names=["circle", "square", "triangle"], output_dict=True, zero_division=0)
(OUT / "metrics.json").write_text(json.dumps({"selected": chosen, "report": report, "test_used": False}, indent=2, ensure_ascii=False), encoding="utf-8")

# 행=실제, 열=예측
confusion = pd.DataFrame(confusion_matrix(chosen_pred.actual, chosen_pred.predicted, labels=[0, 1, 2]))
confusion.to_csv(OUT / "confusion.csv", index=False)
print(metrics_table.drop(columns="seconds").round(4).to_string(index=False))
confusion

 accuracy  macro_f1  circle_recall  square_recall  triangle_recall   loss strategy balance  best_epoch  trainable  epochs           source
      1.0       1.0            1.0            1.0              1.0 0.0126     head    none           8        771       8 synthetic_source
      1.0       1.0            1.0            1.0              1.0 0.0081  partial    none           8       1955       8 synthetic_source
      1.0       1.0            1.0            1.0              1.0 0.0066     head  weight           8        771       8 synthetic_source
      1.0       1.0            1.0            1.0              1.0 0.0065     head sampler           8        771       8 synthetic_source


,0,1,2
0,30,0,0
1,0,30,0
2,0,0,30


## 5단계 · 오류·CAM 저장
오답이 0건이면 error 이미지와 cam_wrong은 만들지 않는다.

In [6]:
# 제공 설명 도구
from explain import save_explanations

explanations = save_explanations(DATA, OUT, model, chosen_pred)
(OUT / "cam_index.json").write_text(json.dumps(explanations, ensure_ascii=False, indent=2), encoding="utf-8")
print("오답 수:", int((chosen_pred.actual != chosen_pred.predicted).sum()))
explanations

오답 수: 0


[{'kind': 'correct', 'path': 'valid/circle/circle_000.png', 'target_class': 0}]

## 6단계 · 모델과 계약 저장 (TODO D09)
복원 시 `weights_only=True`로 읽으므로 텐서·dict·list·str·숫자만 넣는다.

In [7]:
# TODO D09 정답
torch.save({"model": chosen_state,
            "class_to_idx": train.class_to_idx,
            "input": {"color": "RGB", "size": [32, 32], "transform": "Resize((32,32)) -> ToTensor()", "range": "float [0,1]", "normalize": "none"},
            "architecture": "TinyShapeCNN_3-8-16_GAP4 (models.build_model)",
            "selected": chosen,
            "test_used": False}, OUT / "best_model.pt")
print("저장:", (OUT / "best_model.pt").stat().st_size, "bytes")

저장: 14201 bytes


## 7단계 · 복원과 비교 (TODO D10)
새 모델 객체를 만들어 저장 파일로 복원한다. (`restore.py`는 별도 프로세스에서 같은 확인을 한다.)

In [8]:
# 제공 평가 함수·모델 생성
from engine import evaluate
from models import build_model
# valid 배치 공급
from torch.utils.data import DataLoader

saved = torch.load(OUT / "best_model.pt", map_location="cpu", weights_only=True)
restored = build_model(DATA)

# TODO D10 정답
restored.load_state_dict(saved["model"])
restored.eval()

_, y, prob = evaluate(restored, DataLoader(valid, batch_size=32, shuffle=False))
expected = pd.read_csv(OUT / "predictions.csv")
paths = [str(Path(p).relative_to(DATA)) for p, _ in valid.samples]
print("class_to_idx:", saved["class_to_idx"])
print("ID 순서:", paths == expected.path.tolist())
print("확률 일치:", np.allclose(prob, expected[["p0", "p1", "p2"]].to_numpy(), atol=1e-6, rtol=1e-5))
print("예측 일치:", np.array_equal(prob.argmax(1), expected.predicted.to_numpy()))

class_to_idx: {'circle': 0, 'square': 1, 'triangle': 2}
ID 순서: True
확률 일치: True
예측 일치: True


## 8단계 · expected 대조

In [9]:
for name in ["metrics.csv", "history.csv", "predictions.csv", "confusion.csv"]:
    a = pd.read_csv(BASE / "expected" / name)
    b = pd.read_csv(OUT / name)
    num = [c for c in a.columns if pd.api.types.is_numeric_dtype(a[c]) and c != "seconds"]
    print(name, "일치:", np.allclose(a[num], b[num], atol=1e-6))
print("cam_index 일치:", json.loads((BASE / "expected" / "cam_index.json").read_text()) == explanations)

metrics.csv 일치: True
history.csv 일치: True


predictions.csv 일치: True
confusion.csv 일치: True
cam_index 일치: True


## 정리
- 네 후보 모두 valid Macro F1 1.0 → valid loss 최소인 head·sampler 선택. 2위와 차이가 매우 작아 우열은 확정적이지 않다.
- 최종 모델 오답 0건 → 오류 분석은 실습05 초기 모델과 구분한다.
- 저장 파일에 가중치·class_to_idx·입력 계약을 함께 넣어 복원 3항목이 True였다.
- 같은 valid로 반복 선택했으므로 성능은 낙관적이다. test는 선택 후 1회만 평가한다.